# CSE258R Assignment 2 | Food.com Cold-Start Recipe Rating Prediction

**Research question:** For users with rating history, can we predict the star rating they would give to a recipe with no prior ratings, using ingredients and personal taste?

This notebook contains the **full training pipeline code** used in the assignment, plus saved experimental results.

| Section | What is included |
| --- | --- |
| 1. Data preparation | Full cleaning/split script (`wyt/prepare_foodcom.py`) |
| 2. Baseline models | Full baseline model + tuning/eval code |
| 3. Personalization | Features, Models A/B/C, full tuning/test loops |
| 4. Results | Metrics loaded from saved outputs (always runs) |

Set `RUN_FULL_PIPELINE = True` only if you want to re-run expensive training. Default is `False`: code is present and defined, but long jobs are skipped; results come from `baseline/outputs` and `personalization/outputs`.

**Primary metric: MSE** (lower is better).

## 0. Setup

In [ ]:
from pathlib import Path
import ast
import csv
import json
import re
import time
import itertools
from collections import Counter
from datetime import date
from itertools import product

import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

# False = show/define all training code but skip expensive re-runs; load saved metrics instead.
RUN_FULL_PIPELINE = False

pd.set_option("display.precision", 4)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)


def find_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "data_prepared" / "shared").is_dir():
            return p
    raise FileNotFoundError("data_prepared/shared not found: unzip shared data at repo root")


ROOT = find_root()
SHARED = ROOT / "data_prepared" / "shared"
WYT_DATA = ROOT / "data_prepared" / "wyt"
BASE_OUT = ROOT / "baseline" / "outputs"
PERS_OUT = ROOT / "personalization" / "outputs"
TEST_LABELS = WYT_DATA / "test_labels.csv"
print("ROOT =", ROOT)
print("RUN_FULL_PIPELINE =", RUN_FULL_PIPELINE)

try:
    display  # noqa: F821  # provided by Jupyter
except NameError:
    def display(obj):
        if isinstance(obj, pd.DataFrame):
            print(obj.to_string())
        else:
            print(obj)

---
## 1. Data preparation

Source: [Food.com Recipes and User Interactions](https://www.kaggle.com/shuyangli94/food-com-recipes-and-user-interactions).

### 1.1 Protocol

1. Clean recipes/interactions; keep ratings in {1,...,5}; drop pre-publication ratings and 0-star rows.
2. Chronological split: train before 2010-04-25; middle 2010-04-25 to 2012-09-10; late from 2012-09-11.
3. Cold-start eval rows: user has prior history, rating date equals the recipe's first valid rating day.
4. Validation scores only `is_cold_start_eval=1`. Final test refits on train+all middle, predicts late cold-start rows.

### 1.2 Full cleaning and split code (`wyt/prepare_foodcom.py`)

In [ ]:
"""Clean Food.com and export chronological first-rating-day evaluation splits."""

import ast
import csv
import json
import re
from collections import Counter
from datetime import date
from pathlib import Path

ROOT = find_root()  # notebook: reuse Setup cell
VALIDATION_START = "2010-04-25"
TEST_START = "2012-09-11"
COLUMNS = ("row_id", "user_id", "recipe_id", "date", "rating", "ingredients", "submitted")


def valid_date(value):
    try:
        return date.fromisoformat(value).isoformat() == value
    except (TypeError, ValueError):
        return False


def main():
    shared = ROOT / "data_prepared" / "shared"
    wyt = ROOT / "data_prepared" / "wyt"
    shared.mkdir(parents=True, exist_ok=True)
    wyt.mkdir(parents=True, exist_ok=True)
    recipes, seen_ids, recipe_counts = {}, set(), Counter()
    with (ROOT / "RAW_recipes.csv").open(encoding="utf-8-sig", newline="") as source:
        for row in csv.DictReader(source):
            recipe_counts["raw"] += 1
            recipe_id = row["id"].strip()
            if not recipe_id:
                recipe_counts["missing_id"] += 1
                continue
            if recipe_id in seen_ids:
                raise ValueError(f"Duplicate recipe ID: {recipe_id}")
            seen_ids.add(recipe_id)
            if not valid_date(row["submitted"]):
                recipe_counts["invalid_submission_date"] += 1
                continue
            try:
                ingredients = ast.literal_eval(row["ingredients"])
            except (SyntaxError, ValueError):
                ingredients = None
            if not isinstance(ingredients, list) or not all(isinstance(i, str) for i in ingredients):
                recipe_counts["invalid_ingredient_list"] += 1
                continue
            ingredients = [re.sub(r"\s+", " ", i).strip().lower() for i in ingredients]
            ingredients = [i for i in ingredients if i]
            if not ingredients:
                recipe_counts["empty_ingredients"] += 1
                continue
            recipes[recipe_id] = (json.dumps(ingredients, ensure_ascii=False), row["submitted"])
            recipe_counts["retained"] += 1

    rows, first_day, cleaning, rating_counts = [], {}, Counter(), Counter()
    with (ROOT / "RAW_interactions.csv").open(encoding="utf-8-sig", newline="") as source:
        for row_id, row in enumerate(csv.DictReader(source), start=1):
            cleaning["raw"] += 1
            rating = row["rating"].strip()
            rating_counts[rating] += 1
            if rating == "0":
                cleaning["zero_rating"] += 1
                continue
            if rating not in {"1", "2", "3", "4", "5"}:
                cleaning["invalid_rating"] += 1
                continue
            user, recipe, day = row["user_id"].strip(), row["recipe_id"].strip(), row["date"]
            if not user or not recipe or not valid_date(day):
                cleaning["invalid_id_or_date"] += 1
                continue
            if recipe not in recipes:
                cleaning["recipe_missing_or_unusable"] += 1
                continue
            if day < recipes[recipe][1]:
                cleaning["rating_before_publication"] += 1
                continue
            rows.append((row_id, user, recipe, day, int(rating)))
            first_day[recipe] = min(first_day.get(recipe, day), day)
            cleaning["retained"] += 1

    assert cleaning["raw"] == sum(v for k, v in cleaning.items() if k != "raw")
    assert recipe_counts["raw"] == sum(v for k, v in recipe_counts.items() if k != "raw")
    rows.sort(key=lambda row: (row[3], row[0]))
    train = [r for r in rows if r[3] < VALIDATION_START]
    middle = [r for r in rows if VALIDATION_START <= r[3] < TEST_START]
    late = [r for r in rows if r[3] >= TEST_START]
    assert len(train) + len(middle) + len(late) == len(rows)
    train_users = Counter(r[1] for r in train)
    final_users = train_users + Counter(r[1] for r in middle)
    validation_ids = {r[0] for r in middle if r[1] in train_users and r[3] == first_day[r[2]]}
    validation = [r for r in middle if r[0] in validation_ids]
    test = [r for r in late if r[1] in final_users and r[3] == first_day[r[2]]]
    assert {r[2] for r in validation}.isdisjoint(r[2] for r in train)
    assert {r[2] for r in test}.isdisjoint(r[2] for r in train + middle)

    def write(path, header, values):
        with path.open("w", encoding="utf-8", newline="") as dest:
            writer = csv.writer(dest)
            writer.writerow(header)
            writer.writerows(values)

    def enrich(row):
        return (*row, *recipes[row[2]])

    write(wyt / "ratings_clean.csv", COLUMNS, (enrich(r) for r in rows))
    write(shared / "train.csv", COLUMNS, (enrich(r) for r in train))
    write(shared / "validation.csv", (*COLUMNS, "is_cold_start_eval"),
          ((*enrich(r), int(r[0] in validation_ids)) for r in middle))
    write(shared / "test_features.csv", (*COLUMNS[:4], *COLUMNS[5:]),
          ((*r[:4], *recipes[r[2]]) for r in test))
    write(wyt / "test_labels.csv", ("row_id", "rating"), ((r[0], r[4]) for r in test))

    def stats(records):
        return {"ratings": len(records), "users": len({r[1] for r in records}),
                "recipes": len({r[2] for r in records}),
                "first_date": records[0][3], "last_date": records[-1][3],
                "rating_distribution": dict(sorted(Counter(r[4] for r in records).items()))}

    report = {
        "validation_start": VALIDATION_START, "test_start": TEST_START,
        "recipe_cleaning": dict(recipe_counts), "interaction_cleaning": dict(cleaning),
        "raw_rating_distribution": dict(sorted(rating_counts.items())),
        "splits": {name: stats(records) for name, records in [
            ("all_clean", rows), ("train", train), ("validation_all", middle),
            ("validation_first_day", validation), ("test_all", late), ("test_first_day", test)]},
    }
    (ROOT / "data_prepared" / "data_summary.json").write_text(
        json.dumps(report, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(report, indent=2))


if RUN_FULL_PIPELINE:
    main()
else:
    print('Skipping data rebuild (RUN_FULL_PIPELINE=False). Using data_prepared/.')

In [ ]:
summary = json.loads((ROOT / "data_prepared" / "data_summary.json").read_text(encoding="utf-8"))
cleaning = pd.Series(summary["interaction_cleaning"], name="count").to_frame()
cleaning["share"] = cleaning["count"] / cleaning.loc["raw", "count"]
print("validation_start =", summary["validation_start"], "| test_start =", summary["test_start"])
display(cleaning)

_rows = []
for name, key in [
    ("train", "train"),
    ("validation_all (middle)", "validation_all"),
    ("validation cold-start", "validation_first_day"),
    ("test_all (late)", "test_all"),
    ("test cold-start", "test_first_day"),
]:
    s = summary["splits"][key]
    dist = s["rating_distribution"]
    _rows.append({
        "split": name,
        "ratings": s["ratings"],
        "users": s["users"],
        "recipes": s["recipes"],
        "first_date": s["first_date"],
        "last_date": s["last_date"],
        "share_5star": dist.get("5", 0) / s["ratings"],
    })
display(pd.DataFrame(_rows).set_index("split"))

---
## 2. Baseline models

### 2.1 Model list

| Model | Method | Tuning |
| --- | --- | --- |
| `global_mean` | Training-set mean rating | none |
| `global_median` | Training-set median (5 here) | none |
| `user_mean` | `(sum(r) + k * mu) / (n + k)` | k=5 by validation MSE |
| `user_median` | Add k pseudo-counts of the global median, then median | k=2 by validation MAE |
| `ridge_tfidf` | Ingredient TF-IDF (word, min_df=5) + Ridge | alpha=300 by validation MSE |
| `ridge_tfidf_shifted` | Add constant shift, clip to [1, 5] | shift +0.65 by validation MAE |

### 2.2 Full baseline code (from `baseline/baseline.ipynb`)

In [ ]:
# Baseline constants / IO (adapted from baseline/baseline.ipynb setup)
OUT = BASE_OUT
PRED_DIR, METRIC_DIR, TUNING_DIR = OUT / "predictions", OUT / "metrics", OUT / "tuning"
STARS = np.arange(1, 6)
USER_K_GRID = [0, 1, 2, 3, 5, 10, 20, 50]
ANALYZERS = {
    "whole": lambda ingredients: ingredients,
    "word": lambda ingredients: [w for i in ingredients for w in i.split()],
}
RIDGE_GRID = {"analyzer": ["whole", "word"], "min_df": [2, 5, 20], "alpha": [10, 30, 100, 300, 1000]}
SHIFTS = np.round(np.arange(-1, 1.001, 0.01), 2)
TEST_FIRST_YEAR_END = "2013-09-10"
PERIOD_LABELS = ["2012-09-11..2013-09-10", "2013-09-11..2018-12-17"]
TUNED_ON = {
    "global_mean": "-", "global_median": "-", "user_mean": "MSE (k)", "user_median": "MAE (k)",
    "ridge_tfidf": "MSE", "ridge_tfidf_shifted": "MSE + MAE (shift)",
}


def read_split(name):
    df = pd.read_csv(SHARED / f"{name}.csv", dtype={"user_id": str, "recipe_id": str})
    parsed = {s: json.loads(s) for s in df["ingredients"].unique()}
    df["ingredients"] = df["ingredients"].map(parsed)
    return df


def load_baseline_stage(stage):
    train = read_split("train")
    if stage == "val":
        validation = read_split("validation")
        eval_df = validation[validation["is_cold_start_eval"] == 1].reset_index(drop=True)
        return train, eval_df
    validation = read_split("validation").drop(columns="is_cold_start_eval")
    fit_df = pd.concat([train, validation], ignore_index=True)
    eval_df = read_split("test_features")
    return fit_df, eval_df

In [ ]:
def mae(y, p):
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(p))))


def mse(y, p):
    return float(np.mean((np.asarray(y) - np.asarray(p)) ** 2))


def bootstrap_ci(row_loss, users, n_boot=1000, seed=0):
    """95% CI for the mean of a per-row loss, resampling users (not rows)."""
    loss = pd.Series(np.asarray(row_loss)).groupby(np.asarray(users))
    sums, counts = loss.sum().to_numpy(), loss.count().to_numpy()
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(sums), size=(n_boot, len(sums)))
    boots = sums[idx].sum(axis=1) / counts[idx].sum(axis=1)
    return float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def test_period(eval_df):
    """Label each test row as first year after the test start or later."""
    first = (eval_df["date"] <= TEST_FIRST_YEAR_END).to_numpy()
    return pd.Categorical(np.where(first, *PERIOD_LABELS), categories=PERIOD_LABELS)


def attach_test_labels(eval_df):
    labels = pd.read_csv(TEST_LABELS)
    assert labels["row_id"].is_unique, "duplicate row_id in test_labels.csv"
    merged = eval_df.merge(labels, on="row_id", how="left", validate="one_to_one")
    assert merged["rating"].notna().all(), "some test rows have no label"
    assert len(labels) == len(eval_df), "test_labels.csv has rows without a prediction"
    return merged


def write_predictions(eval_df, preds, name, stage):
    out = pd.DataFrame({"row_id": eval_df["row_id"], "prediction": preds})
    assert out["row_id"].is_unique and out["prediction"].notna().all()
    out.to_csv(PRED_DIR / f"{stage}_{name}.csv", index=False)


def summarize(name, y, p, users):
    abs_err, sq_err = np.abs(p - y), (p - y) ** 2
    (mae_lo, mae_hi), (mse_lo, mse_hi) = bootstrap_ci(abs_err, users), bootstrap_ci(sq_err, users)
    return {"model": name, "tuned_on": TUNED_ON[name],
            "MAE": abs_err.mean(), "MAE_95CI": f"[{mae_lo:.4f}, {mae_hi:.4f}]",
            "MSE": sq_err.mean(), "MSE_95CI": f"[{mse_lo:.4f}, {mse_hi:.4f}]"}


def by_group(name, y, p, eval_df, groups, group_col):
    groups = pd.Categorical(groups)
    rows = []
    for label in groups.categories:
        m = np.asarray(groups == label)
        rows.append({"model": name, group_col: label, "rows": int(m.sum()),
                     "users": eval_df.loc[m, "user_id"].nunique(),
                     "MAE": mae(y[m], p[m]), "MSE": mse(y[m], p[m])})
    return rows


def score(preds, y, eval_df):
    """Overall MAE / MSE of every model, with user-bootstrap CIs."""
    return pd.DataFrame([summarize(n, y, p, eval_df["user_id"]) for n, p in preds.items()])


def score_by(preds, y, eval_df, groups, group_col):
    """Long table of MAE / MSE per model and group."""
    return pd.DataFrame([r for n, p in preds.items() for r in by_group(n, y, p, eval_df, groups, group_col)])


def wide(grouped, group_col, order):
    """Pivot a long per-group table to models x (metric, group) for display."""
    table = grouped.pivot(index="model", columns=group_col, values=["MAE", "MSE"])
    return table.reindex(index=order, columns=pd.MultiIndex.from_product([["MAE", "MSE"], order_of(grouped, group_col)]))


def order_of(grouped, group_col):
    return list(dict.fromkeys(grouped[group_col]))

In [ ]:
def user_star_counts(fit_df):
    """users x 5 matrix: how many 1..5-star ratings each user gave in the fitting data."""
    return pd.crosstab(fit_df["user_id"], fit_df["rating"]).reindex(columns=STARS, fill_value=0)


def predict_user_mean(counts, users, global_mean, k):
    c = counts.to_numpy(float)
    est = (c @ STARS + k * global_mean) / (c.sum(axis=1) + k)
    return pd.Series(est, index=counts.index).reindex(users).fillna(global_mean).to_numpy()


def predict_user_median(counts, users, global_median, k):
    c = counts.to_numpy(float).copy()
    c[:, int(global_median) - 1] += k
    n, cum = c.sum(axis=1), c.cumsum(axis=1)
    lo, hi = (n - 1) // 2, n // 2  # 0-based middle positions (equal when n is odd)
    v_lo = (cum > lo[:, None]).argmax(axis=1) + 1
    v_hi = (cum > hi[:, None]).argmax(axis=1) + 1
    est = (v_lo + v_hi) / 2  # same convention as np.median for even counts
    return pd.Series(est, index=counts.index).reindex(users).fillna(global_median).to_numpy()


def fit_ridge(fit_df, eval_df, analyzer, min_df, alpha):
    vec = TfidfVectorizer(analyzer=ANALYZERS[analyzer], min_df=min_df)
    vec.fit(fit_df.drop_duplicates("recipe_id")["ingredients"])
    model = Ridge(alpha=alpha, solver="sparse_cg")
    model.fit(vec.transform(fit_df["ingredients"]), fit_df["rating"].to_numpy(float))
    return vec, model, model.predict(vec.transform(eval_df["ingredients"]))


def predict_all(fit_df, eval_df, params, ridge_pred=None):
    """Predictions of every baseline given tuned params (shared by the val and test stages)."""
    y_fit, users = fit_df["rating"].to_numpy(float), eval_df["user_id"]
    g_mean, g_median = y_fit.mean(), float(np.median(y_fit))
    counts = user_star_counts(fit_df)
    if ridge_pred is None:
        r = params["ridge"]
        ridge_pred = fit_ridge(fit_df, eval_df, r["analyzer"], r["min_df"], r["alpha"])[2]
    return {
        "global_mean": np.full(len(eval_df), g_mean),
        "global_median": np.full(len(eval_df), g_median),
        "user_mean": predict_user_mean(counts, users, g_mean, params["user_mean_k"]),
        "user_median": predict_user_median(counts, users, g_median, params["user_median_k"]),
        "ridge_tfidf": ridge_pred,
        "ridge_tfidf_shifted": np.clip(ridge_pred + params["ridge_shift"], 1, 5),
    }

In [ ]:
# Full baseline validation tuning + test scoring (expensive). Skipped unless RUN_FULL_PIPELINE.
if RUN_FULL_PIPELINE:
    for d in (PRED_DIR, METRIC_DIR, TUNING_DIR):
        d.mkdir(parents=True, exist_ok=True)

    fit_df, eval_df = load_baseline_stage("val")
    y_fit = fit_df["rating"].to_numpy(float)
    y = eval_df["rating"].to_numpy(float)
    g_mean, g_median = float(y_fit.mean()), float(np.median(y_fit))
    counts = user_star_counts(fit_df)
    users = eval_df["user_id"]

    user_grid = []
    for k in USER_K_GRID:
        pm = predict_user_mean(counts, users, g_mean, k)
        pmed = predict_user_median(counts, users, g_median, k)
        user_grid.append({
            "k": k,
            "user_mean_MSE": mse(y, pm),
            "user_mean_MAE": mae(y, pm),
            "user_median_MAE": mae(y, pmed),
            "user_median_MSE": mse(y, pmed),
        })
    user_grid = pd.DataFrame(user_grid)
    params = {
        "user_mean_k": int(user_grid.loc[user_grid["user_mean_MSE"].idxmin(), "k"]),
        "user_median_k": int(user_grid.loc[user_grid["user_median_MAE"].idxmin(), "k"]),
    }

    ridge_rows = []
    best = None
    for analyzer, min_df, alpha in itertools.product(
        RIDGE_GRID["analyzer"], RIDGE_GRID["min_df"], RIDGE_GRID["alpha"]
    ):
        vec, model, p = fit_ridge(fit_df, eval_df, analyzer, min_df, alpha)
        row = {
            "analyzer": analyzer, "min_df": min_df, "alpha": alpha,
            "vocab": len(vec.vocabulary_), "MSE": mse(y, p), "MAE": mae(y, p),
            "pred_std": float(np.std(p)),
        }
        ridge_rows.append(row)
        if best is None or row["MSE"] < best[0]:
            best = (row["MSE"], analyzer, min_df, alpha, p)
    params["ridge"] = {"analyzer": best[1], "min_df": best[2], "alpha": float(best[3])}
    ridge_pred = best[4]
    curve = pd.DataFrame({
        "shift": SHIFTS,
        "MAE": [mae(y, np.clip(ridge_pred + c, 1, 5)) for c in SHIFTS],
    })
    params["ridge_shift"] = float(curve.loc[curve["MAE"].idxmin(), "shift"])

    (TUNING_DIR / "params.json").write_text(json.dumps(params, indent=2) + "\n", encoding="utf-8")
    preds = predict_all(fit_df, eval_df, params, ridge_pred)
    # write validation predictions / metrics via helpers already defined in baseline cells
    print("baseline val params:", params)
    print(score(preds, y, eval_df))

    fit_df, eval_df = load_baseline_stage("test")
    labels = pd.read_csv(TEST_LABELS)
    eval_df = eval_df.merge(labels, on="row_id", how="left", validate="one_to_one")
    test_preds = predict_all(fit_df, eval_df, params)
    y_test = eval_df["rating"].to_numpy(float)
    print(score(test_preds, y_test, eval_df))
else:
    print("Skipping baseline retrain (RUN_FULL_PIPELINE=False). Loading saved metrics.")

In [ ]:
base_val = pd.read_csv(BASE_OUT / "metrics" / "val_metrics.csv")
base_test = pd.read_csv(BASE_OUT / "metrics" / "test_metrics.csv")
base_period = pd.read_csv(BASE_OUT / "metrics" / "test_metrics_by_period.csv")
base_params = json.loads((BASE_OUT / "tuning" / "params.json").read_text(encoding="utf-8"))
print("baseline tuned params:")
print(json.dumps(base_params, indent=2))

def show_metrics(df, title):
    cols = [c for c in ["model", "tuned_on", "MAE", "MAE_95CI", "MSE", "MSE_95CI"] if c in df.columns]
    print(title)
    return df[cols].copy()

display(show_metrics(base_val, "Validation (cold-start 24,252)"))
display(show_metrics(base_test, "Test (cold-start 11,668)"))
display(base_period.pivot(index="model", columns="period", values="MSE").reindex(columns=[
    "overall", "2012-09-11..2013-09-10", "2013-09-11..2018-12-17"
]))

---
## 3. Personalization models

### 3.1 Design

Leakage-safe features per sample (u, i, t): `user_mean`, taste-similarity `sim`, history length, optional `global_ridge`.

| Family | Idea |
| --- | --- |
| A | Ridge on absolute rating |
| B | Hard user base + linear residual |
| C (MSE submit) | Hard `user_mean` + HistGradientBoosting residual |

### 3.2 Feature engineering (`personalization/features.py`)

In [ ]:
"""Leakage-safe user/content features for personalized rating models."""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

STARS = np.arange(1, 6, dtype=float)


def find_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / "data_prepared" / "shared").is_dir():
            return p
    raise FileNotFoundError("data_prepared/shared not found: unzip shared data at repo root")


def word_analyzer(ingredients: list[str]) -> list[str]:
    return [w for item in ingredients for w in item.split()]


def read_split(shared: Path, name: str) -> pd.DataFrame:
    df = pd.read_csv(shared / f"{name}.csv", dtype={"user_id": str, "recipe_id": str})
    parsed = {s: json.loads(s) for s in df["ingredients"].unique()}
    df["ingredients"] = df["ingredients"].map(parsed)
    return df


def load_stage(shared: Path, stage: str):
    """stage='val' uses train fit + cold-start validation rows; 'test' merges train+val."""
    train = read_split(shared, "train")
    if stage == "val":
        validation = read_split(shared, "validation")
        eval_df = validation.loc[validation["is_cold_start_eval"] == 1].reset_index(drop=True)
        return train, eval_df
    if stage == "test":
        validation = read_split(shared, "validation").drop(columns="is_cold_start_eval")
        fit_df = pd.concat([train, validation], ignore_index=True)
        eval_df = read_split(shared, "test_features")
        return fit_df, eval_df
    raise ValueError(stage)


def shrunk_user_mean(sum_r: float, n: int, global_mean: float, k: float) -> float:
    if n <= 0:
        return float(global_mean)
    return float((sum_r + k * global_mean) / (n + k))


def shrunk_user_median(counts: np.ndarray, global_median: float, k: float) -> float:
    c = counts.astype(float).copy()
    c[int(global_median) - 1] += k
    n = c.sum()
    if n <= 0:
        return float(global_median)
    cum = np.cumsum(c)
    lo, hi = int((n - 1) // 2), int(n // 2)
    v_lo = int(np.argmax(cum > lo) + 1)
    v_hi = int(np.argmax(cum > hi) + 1)
    return 0.5 * (v_lo + v_hi)


class ContentEncoder:
    """TF-IDF (word) + SVD recipe vectors, plus a global ingredient Ridge score."""

    def __init__(self, n_components: int = 128, min_df: int = 5, ridge_alpha: float = 300.0):
        self.n_components = n_components
        self.min_df = min_df
        self.ridge_alpha = ridge_alpha
        self.vectorizer = TfidfVectorizer(analyzer=word_analyzer, min_df=min_df)
        self.svd = TruncatedSVD(n_components=n_components, random_state=0)
        self.ridge = Ridge(alpha=ridge_alpha, solver="sparse_cg")
        self.recipe_vec: dict[str, np.ndarray] = {}

    def fit(self, fit_df: pd.DataFrame) -> "ContentEncoder":
        recipes = fit_df.drop_duplicates("recipe_id")
        tfidf_recipes = self.vectorizer.fit_transform(recipes["ingredients"])
        low = self.svd.fit_transform(tfidf_recipes)
        self.recipe_vec = {rid: low[i] for i, rid in enumerate(recipes["recipe_id"].to_numpy())}

        tfidf_rows = self.vectorizer.transform(fit_df["ingredients"])
        self.ridge.fit(tfidf_rows, fit_df["rating"].to_numpy(dtype=float))
        return self

    def transform_recipes(self, recipe_ids: pd.Series) -> np.ndarray:
        dim = self.n_components
        out = np.zeros((len(recipe_ids), dim), dtype=np.float64)
        lookup = self.recipe_vec
        for i, rid in enumerate(recipe_ids.to_numpy()):
            vec = lookup.get(rid)
            if vec is not None:
                out[i] = vec
            # unseen cold-start recipes: encode from ingredients via vectorizer+SVD below if needed
        return out

    def transform_ingredients(self, ingredients: pd.Series) -> np.ndarray:
        return self.svd.transform(self.vectorizer.transform(ingredients))

    def recipe_matrix(self, df: pd.DataFrame) -> np.ndarray:
        """Prefer fitted recipe vectors; fall back to transforming ingredients for new recipes."""
        known = df["recipe_id"].isin(self.recipe_vec)
        out = np.zeros((len(df), self.n_components), dtype=np.float64)
        if known.any():
            out[known.to_numpy()] = self.transform_recipes(df.loc[known, "recipe_id"])
        if (~known).any():
            out[(~known).to_numpy()] = self.transform_ingredients(df.loc[~known, "ingredients"])
        return out

    def global_ridge_predict(self, ingredients: pd.Series) -> np.ndarray:
        return self.ridge.predict(self.vectorizer.transform(ingredients))


def _similarity(sum_rx: np.ndarray, sum_x: np.ndarray, mu: float, x: np.ndarray, n: int, sim_k: float) -> float:
    if n <= 0:
        return 0.0
    p = sum_rx - mu * sum_x
    pn = float(np.linalg.norm(p))
    xn = float(np.linalg.norm(x))
    if pn < 1e-12 or xn < 1e-12:
        s = 0.0
    else:
        s = float(np.dot(p, x) / (pn * xn))
    return s * (n / (n + sim_k)) if sim_k > 0 else s


def build_prefix_feature_table(
    df: pd.DataFrame,
    X: np.ndarray,
    global_mean: float,
    global_median: float,
    user_mean_k: float,
    user_median_k: float,
    sim_k: float,
    global_ridge: np.ndarray | None = None,
) -> pd.DataFrame:
    """Per-row features using only strictly earlier ratings of the same user."""
    order = np.lexsort((df["row_id"].to_numpy(), df["date"].to_numpy(), df["user_id"].to_numpy()))
    users = df["user_id"].to_numpy()[order]
    ratings = df["rating"].to_numpy(dtype=float)[order]
    Xo = X[order]
    ridge = None if global_ridge is None else global_ridge[order]

    n_rows, dim = Xo.shape
    user_mean = np.empty(n_rows, dtype=np.float64)
    user_median = np.empty(n_rows, dtype=np.float64)
    sim = np.empty(n_rows, dtype=np.float64)
    log_n = np.empty(n_rows, dtype=np.float64)
    n_hist = np.empty(n_rows, dtype=np.int32)

    i = 0
    while i < n_rows:
        j = i + 1
        while j < n_rows and users[j] == users[i]:
            j += 1
        sum_x = np.zeros(dim, dtype=np.float64)
        sum_rx = np.zeros(dim, dtype=np.float64)
        counts = np.zeros(5, dtype=np.float64)
        sum_r = 0.0
        n = 0
        for t in range(i, j):
            mu = shrunk_user_mean(sum_r, n, global_mean, user_mean_k)
            med = shrunk_user_median(counts, global_median, user_median_k) if n > 0 else float(global_median)
            user_mean[t] = mu
            user_median[t] = med
            sim[t] = _similarity(sum_rx, sum_x, mu, Xo[t], n, sim_k)
            log_n[t] = np.log1p(n)
            n_hist[t] = n

            r = ratings[t]
            sum_x += Xo[t]
            sum_rx += r * Xo[t]
            sum_r += r
            counts[int(r) - 1] += 1.0
            n += 1
        i = j

    inv = np.empty(n_rows, dtype=np.int64)
    inv[order] = np.arange(n_rows)
    out = {
        "user_mean": user_mean[inv],
        "user_median": user_median[inv],
        "sim": sim[inv],
        "log_n": log_n[inv],
        "n_hist": n_hist[inv],
    }
    if ridge is not None:
        out["global_ridge"] = ridge[inv]
    return pd.DataFrame(out)


def build_user_profiles(
    fit_df: pd.DataFrame,
    X_fit: np.ndarray,
    global_mean: float,
    global_median: float,
    user_mean_k: float,
    user_median_k: float,
) -> dict[str, dict]:
    """Freeze one profile per user from the full fitting history."""
    order = np.lexsort((fit_df["row_id"].to_numpy(), fit_df["date"].to_numpy(), fit_df["user_id"].to_numpy()))
    users = fit_df["user_id"].to_numpy()[order]
    ratings = fit_df["rating"].to_numpy(dtype=float)[order]
    Xo = X_fit[order]
    profiles: dict[str, dict] = {}

    i = 0
    n_rows, dim = Xo.shape
    while i < n_rows:
        j = i + 1
        while j < n_rows and users[j] == users[i]:
            j += 1
        block = Xo[i:j]
        r = ratings[i:j]
        n = j - i
        sum_r = float(r.sum())
        counts = np.zeros(5, dtype=np.float64)
        for star in r:
            counts[int(star) - 1] += 1.0
        mu = shrunk_user_mean(sum_r, n, global_mean, user_mean_k)
        med = shrunk_user_median(counts, global_median, user_median_k)
        sum_x = block.sum(axis=0)
        sum_rx = (block * r[:, None]).sum(axis=0)
        p = sum_rx - mu * sum_x
        pn = float(np.linalg.norm(p))
        if pn > 1e-12:
            p = p / pn
        else:
            p = np.zeros(dim, dtype=np.float64)
        profiles[users[i]] = {
            "user_mean": mu,
            "user_median": med,
            "taste": p,
            "n": n,
            "log_n": float(np.log1p(n)),
        }
        i = j
    return profiles


def build_eval_feature_table(
    eval_df: pd.DataFrame,
    X_eval: np.ndarray,
    profiles: dict[str, dict],
    global_mean: float,
    global_median: float,
    sim_k: float,
    global_ridge: np.ndarray | None = None,
) -> pd.DataFrame:
    n_rows = len(eval_df)
    user_mean = np.full(n_rows, global_mean, dtype=np.float64)
    user_median = np.full(n_rows, global_median, dtype=np.float64)
    sim = np.zeros(n_rows, dtype=np.float64)
    log_n = np.zeros(n_rows, dtype=np.float64)
    n_hist = np.zeros(n_rows, dtype=np.int32)
    users = eval_df["user_id"].to_numpy()

    for i, uid in enumerate(users):
        prof = profiles.get(uid)
        if prof is None:
            continue
        user_mean[i] = prof["user_mean"]
        user_median[i] = prof["user_median"]
        log_n[i] = prof["log_n"]
        n = prof["n"]
        n_hist[i] = n
        x = X_eval[i]
        xn = float(np.linalg.norm(x))
        if n > 0 and xn > 1e-12:
            s = float(np.dot(prof["taste"], x) / xn)
            sim[i] = s * (n / (n + sim_k)) if sim_k > 0 else s

    out = {
        "user_mean": user_mean,
        "user_median": user_median,
        "sim": sim,
        "log_n": log_n,
        "n_hist": n_hist,
    }
    if global_ridge is not None:
        out["global_ridge"] = global_ridge
    return pd.DataFrame(out)

### 3.3 Models A/B/C (`personalization/models.py`)

In [ ]:
"""Model A (absolute), B (linear residual), and C (HGB residual) on shared features."""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge

FEATURE_COLS_A = ("user_mean", "sim", "log_n")
FEATURE_COLS_B = ("sim", "log_n")
FEATURE_COLS_C = ("user_mean", "sim", "log_n", "n_hist", "global_ridge")


def _design(frame: pd.DataFrame, cols: tuple[str, ...], use_global_ridge: bool) -> np.ndarray:
    names = list(cols)
    if use_global_ridge:
        names.append("global_ridge")
    return frame[names].to_numpy(dtype=float)


def mae(y: np.ndarray, p: np.ndarray) -> float:
    return float(np.mean(np.abs(y - p)))


def mse(y: np.ndarray, p: np.ndarray) -> float:
    return float(np.mean((y - p) ** 2))


def clip_stars(p: np.ndarray) -> np.ndarray:
    return np.clip(p, 1.0, 5.0)


class ModelA:
    """Absolute-rating Ridge on user_mean + taste similarity (+ optional global ridge)."""

    def __init__(self, alpha: float = 10.0, use_global_ridge: bool = False):
        self.alpha = alpha
        self.use_global_ridge = use_global_ridge
        self.model = Ridge(alpha=alpha, fit_intercept=True)

    def fit(self, feats: pd.DataFrame, y: np.ndarray) -> "ModelA":
        self.model.fit(_design(feats, FEATURE_COLS_A, self.use_global_ridge), y)
        return self

    def predict(self, feats: pd.DataFrame) -> np.ndarray:
        return clip_stars(self.model.predict(_design(feats, FEATURE_COLS_A, self.use_global_ridge)))


class ModelB:
    """Two-stage: hard user base + residual Ridge on similarity features."""

    def __init__(
        self,
        base: str = "user_median",
        alpha: float = 10.0,
        use_global_ridge: bool = False,
        residual_shift: float = 0.0,
    ):
        if base not in {"user_mean", "user_median"}:
            raise ValueError(base)
        self.base = base
        self.alpha = alpha
        self.use_global_ridge = use_global_ridge
        self.residual_shift = residual_shift
        self.model = Ridge(alpha=alpha, fit_intercept=True)

    def fit(self, feats: pd.DataFrame, y: np.ndarray) -> "ModelB":
        base = feats[self.base].to_numpy(dtype=float)
        residual = y - base
        self.model.fit(_design(feats, FEATURE_COLS_B, self.use_global_ridge), residual)
        return self

    def predict(self, feats: pd.DataFrame) -> np.ndarray:
        base = feats[self.base].to_numpy(dtype=float)
        delta = self.model.predict(_design(feats, FEATURE_COLS_B, self.use_global_ridge))
        return clip_stars(base + delta + self.residual_shift)

    def predict_raw_residual(self, feats: pd.DataFrame) -> np.ndarray:
        return self.model.predict(_design(feats, FEATURE_COLS_B, self.use_global_ridge))


def tune_residual_shift(base: np.ndarray, residual: np.ndarray, y: np.ndarray, grid: np.ndarray) -> tuple[float, float]:
    """Pick constant shift on residual that minimizes MAE on validation."""
    best_c, best_mae = 0.0, float("inf")
    for c in grid:
        m = mae(y, clip_stars(base + residual + c))
        if m < best_mae:
            best_mae, best_c = m, float(c)
    return best_c, best_mae


class ModelC:
    """MSE-oriented two-stage model: hard user_mean base + HGB residual.

    Residual features can include user_mean / global_ridge so the tree can learn
    interactions (e.g. when content disagrees with the user level).
    """

    def __init__(
        self,
        max_depth: int = 6,
        learning_rate: float = 0.05,
        max_iter: int = 200,
        min_samples_leaf: int = 50,
        l2_regularization: float = 0.0,
        use_global_ridge: bool = True,
    ):
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.min_samples_leaf = min_samples_leaf
        self.l2_regularization = l2_regularization
        self.use_global_ridge = use_global_ridge
        self.model = HistGradientBoostingRegressor(
            loss="squared_error",
            max_depth=max_depth,
            learning_rate=learning_rate,
            max_iter=max_iter,
            min_samples_leaf=min_samples_leaf,
            l2_regularization=l2_regularization,
            random_state=0,
        )

    def _X(self, feats: pd.DataFrame) -> np.ndarray:
        cols = list(FEATURE_COLS_C) if self.use_global_ridge else [c for c in FEATURE_COLS_C if c != "global_ridge"]
        # engineered interactions for the tree
        base = feats[cols].to_numpy(dtype=float)
        sim = feats["sim"].to_numpy(dtype=float)
        log_n = feats["log_n"].to_numpy(dtype=float)
        user_mean = feats["user_mean"].to_numpy(dtype=float)
        ridge = feats["global_ridge"].to_numpy(dtype=float) if self.use_global_ridge else np.zeros(len(feats))
        extra = np.column_stack(
            [
                sim * log_n,
                ridge - user_mean,
                np.abs(sim),
            ]
        )
        return np.hstack([base, extra])

    def fit(self, feats: pd.DataFrame, y: np.ndarray) -> "ModelC":
        base = feats["user_mean"].to_numpy(dtype=float)
        self.model.fit(self._X(feats), y - base)
        return self

    def predict(self, feats: pd.DataFrame) -> np.ndarray:
        base = feats["user_mean"].to_numpy(dtype=float)
        return clip_stars(base + self.model.predict(self._X(feats)))

### 3.4 Full training loops (A/B then C)

In [ ]:
# --- personalization training utilities (from run_experiment.py / run_model_c.py) ---
PERS_PRED = PERS_OUT / "predictions"
PERS_METRIC = PERS_OUT / "metrics"
PERS_TUNING = PERS_OUT / "tuning"
USER_MEAN_K = 5.0
USER_MEDIAN_K = 2.0
SVD_DIM = 128
SIM_K_GRID = [0.0, 5.0, 20.0]
ALPHA_GRID = [1.0, 10.0, 100.0, 1000.0]
RIDGE_FLAGS = [False, True]
SHIFT_GRID = np.round(np.arange(-0.5, 0.501, 0.05), 2)
BASES_B = ["user_median", "user_mean"]


def _jsonable(obj):
    if isinstance(obj, dict):
        return {k: _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (np.floating, float)):
        return float(obj)
    if isinstance(obj, (np.integer, int)):
        return int(obj)
    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    if isinstance(obj, str):
        return obj
    return obj


def apply_sim_shrink(feats, sim_k):
    out = feats.copy()
    n = out["n_hist"].to_numpy(dtype=float)
    raw = out["sim"].to_numpy(dtype=float)
    out["sim"] = raw if sim_k <= 0 else np.where(n > 0, raw * (n / (n + sim_k)), 0.0)
    return out


def write_pers_preds(eval_df, preds, name, stage):
    path = PERS_PRED / f"{stage}_{name}.csv"
    pd.DataFrame({"row_id": eval_df["row_id"], "prediction": preds}).to_csv(path, index=False)


def prepare_pers_stage(stage):
    fit_df, eval_df = load_stage(SHARED, stage)
    y_fit = fit_df["rating"].to_numpy(dtype=float)
    g_mean, g_median = float(y_fit.mean()), float(np.median(y_fit))
    encoder = ContentEncoder(n_components=SVD_DIM, min_df=5, ridge_alpha=300.0).fit(fit_df)
    X_fit = encoder.recipe_matrix(fit_df)
    ridge_fit = encoder.global_ridge_predict(fit_df["ingredients"])
    train_feats = build_prefix_feature_table(
        fit_df, X_fit, g_mean, g_median, USER_MEAN_K, USER_MEDIAN_K, sim_k=0.0, global_ridge=ridge_fit
    )
    X_eval = encoder.recipe_matrix(eval_df)
    ridge_eval = encoder.global_ridge_predict(eval_df["ingredients"])
    profiles = build_user_profiles(fit_df, X_fit, g_mean, g_median, USER_MEAN_K, USER_MEDIAN_K)
    eval_feats = build_eval_feature_table(
        eval_df, X_eval, profiles, g_mean, g_median, sim_k=0.0, global_ridge=ridge_eval
    )
    y_eval = eval_df["rating"].to_numpy(dtype=float) if "rating" in eval_df.columns else None
    return {
        "fit_df": fit_df, "eval_df": eval_df, "train_feats": train_feats, "eval_feats": eval_feats,
        "y_fit": y_fit, "y_eval": y_eval, "g_mean": g_mean, "g_median": g_median,
    }


def fit_predict_from_config(train_feats, eval_feats, y_tr, cfg):
    tr = apply_sim_shrink(train_feats, float(cfg["sim_k"]))
    ev = apply_sim_shrink(eval_feats, float(cfg["sim_k"]))
    use_ridge = bool(cfg["use_global_ridge"])
    alpha = float(cfg["alpha"])
    if cfg["family"] == "B":
        model = ModelB(
            base=str(cfg["base"]), alpha=alpha, use_global_ridge=use_ridge,
            residual_shift=float(cfg["residual_shift"]),
        )
        return model.fit(tr, y_tr).predict(ev)
    return ModelA(alpha=alpha, use_global_ridge=use_ridge).fit(tr, y_tr).predict(ev)


def fit_predict_c(train_feats, eval_feats, y_tr, cfg):
    tr = apply_sim_shrink(train_feats, float(cfg["sim_k"]))
    ev = apply_sim_shrink(eval_feats, float(cfg["sim_k"]))
    model = ModelC(
        max_depth=int(cfg["max_depth"]),
        learning_rate=float(cfg["learning_rate"]),
        max_iter=int(cfg["max_iter"]),
        min_samples_leaf=int(cfg["min_samples_leaf"]),
        l2_regularization=float(cfg["l2_regularization"]),
        use_global_ridge=bool(cfg["use_global_ridge"]),
    )
    return model.fit(tr, y_tr).predict(ev)

In [ ]:
if RUN_FULL_PIPELINE:
    for d in (PERS_PRED, PERS_METRIC, PERS_TUNING):
        d.mkdir(parents=True, exist_ok=True)

    # ---- Models A/B grid on validation ----
    data = prepare_pers_stage("val")
    y_tr, y_va = data["y_fit"], data["y_eval"]
    rows = []
    for sim_k in SIM_K_GRID:
        tr = apply_sim_shrink(data["train_feats"], sim_k)
        va = apply_sim_shrink(data["eval_feats"], sim_k)
        for use_ridge in RIDGE_FLAGS:
            for alpha in ALPHA_GRID:
                for base in BASES_B:
                    model = ModelB(base=base, alpha=alpha, use_global_ridge=use_ridge)
                    model.fit(tr, y_tr)
                    raw_delta = model.predict_raw_residual(va)
                    shift, _ = tune_residual_shift(va[base].to_numpy(float), raw_delta, y_va, SHIFT_GRID)
                    model.residual_shift = shift
                    pred = model.predict(va)
                    rows.append({
                        "family": "B", "base": base, "sim_k": sim_k, "alpha": alpha,
                        "use_global_ridge": use_ridge, "residual_shift": shift,
                        "MAE": mae(y_va, pred), "MSE": mse(y_va, pred),
                    })
                model_a = ModelA(alpha=alpha, use_global_ridge=use_ridge).fit(tr, y_tr)
                pred_a = model_a.predict(va)
                rows.append({
                    "family": "A", "base": "user_mean_feature", "sim_k": sim_k, "alpha": alpha,
                    "use_global_ridge": use_ridge, "residual_shift": 0.0,
                    "MAE": mae(y_va, pred_a), "MSE": mse(y_va, pred_a),
                })
    grid = pd.DataFrame(rows)
    best_mse = _jsonable(grid.loc[grid["MSE"].idxmin()].to_dict())
    params = {"user_mean_k": USER_MEAN_K, "user_median_k": USER_MEDIAN_K, "svd_dim": SVD_DIM, "best_mse": best_mse}
    grid.to_csv(PERS_TUNING / "val_grid.csv", index=False)

    # ---- Model C grid ----
    c_grid_spec = {
        "sim_k": [0.0, 5.0, 20.0], "max_depth": [4, 6], "learning_rate": [0.05, 0.1],
        "max_iter": [200], "min_samples_leaf": [50], "l2_regularization": [0.0, 1.0],
        "use_global_ridge": [True],
    }
    keys = list(c_grid_spec.keys())
    c_rows = []
    for values in product(*[c_grid_spec[k] for k in keys]):
        cfg = dict(zip(keys, values))
        pred = fit_predict_c(data["train_feats"], data["eval_feats"], y_tr, cfg)
        c_rows.append({**cfg, "MAE": mae(y_va, pred), "MSE": mse(y_va, pred)})
    c_grid = pd.DataFrame(c_rows).sort_values("MSE")
    best_c = _jsonable(c_grid.iloc[0].to_dict())
    params["best_c"] = best_c
    params["primary_metric"] = "MSE"
    params["submit_model"] = "best_c" if best_c["MSE"] < best_mse["MSE"] else "best_mse"
    params["submit_family"] = "C" if params["submit_model"] == "best_c" else "A"
    (PERS_TUNING / "params.json").write_text(json.dumps(params, indent=2) + "\n", encoding="utf-8")
    c_grid.to_csv(PERS_TUNING / "val_model_c_grid.csv", index=False)

    # ---- Test refit ----
    data_te = prepare_pers_stage("test")
    labels = pd.read_csv(TEST_LABELS)
    eval_df = data_te["eval_df"].merge(labels, on="row_id", how="left", validate="one_to_one")
    y = eval_df["rating"].to_numpy(float)
    pred_c = fit_predict_c(data_te["train_feats"], data_te["eval_feats"], data_te["y_fit"], best_c)
    write_pers_preds(eval_df, pred_c, "personalization_best_mse", "test")
    print("personalization test MSE (C):", mse(y, pred_c))
else:
    print("Skipping personalization retrain (RUN_FULL_PIPELINE=False). Loading saved metrics.")

In [ ]:
pers_params = json.loads((PERS_OUT / "tuning" / "params.json").read_text(encoding="utf-8"))
print("submit_model =", pers_params.get("submit_model"), "| family =", pers_params.get("submit_family"))
print(json.dumps(pers_params.get("best_c"), indent=2))

pers_val = pd.read_csv(PERS_OUT / "metrics" / "val_metrics_with_c.csv")
pers_test = pd.read_csv(PERS_OUT / "metrics" / "test_metrics_with_c.csv")
pers_period = pd.read_csv(PERS_OUT / "metrics" / "test_metrics_by_period_with_c.csv")
display(show_metrics(pers_val, "Personalization validation"))
display(show_metrics(pers_test, "Personalization test"))
display(pers_period.pivot(index="model", columns="period", values="MSE"))

submit_pred = pd.read_csv(PERS_OUT / "predictions" / "test_personalization_best_mse.csv")
print("submitted predictions:", len(submit_pred), "mean=", submit_pred["prediction"].mean())
display(submit_pred.head())

---
## 4. Overall comparison (MSE)

In [ ]:
board = []
for _, r in base_test.iterrows():
    board.append({"model": r["model"], "group": "baseline", "test_MSE": r["MSE"], "test_MAE": r["MAE"]})
for _, r in pers_test.iterrows():
    if r["model"] == "ref_user_mean":
        continue
    board.append({"model": r["model"], "group": "personalization", "test_MSE": r["MSE"], "test_MAE": r["MAE"]})
board_df = pd.DataFrame(board).drop_duplicates("model").sort_values("test_MSE").reset_index(drop=True)
um = float(board_df.loc[board_df["model"] == "user_mean", "test_MSE"].iloc[0])
board_df["delta_vs_user_mean"] = board_df["test_MSE"] - um
display(board_df)

um = float(base_test.loc[base_test["model"] == "user_mean", "MSE"].iloc[0])
pc = float(pers_test.loc[pers_test["model"] == "personalization_C", "MSE"].iloc[0])
pa = float(pers_test.loc[pers_test["model"] == "personalization_A", "MSE"].iloc[0])
print(f"user_mean          test MSE = {um:.4f}")
print(f"personalization A  test MSE = {pa:.4f}   (delta {pa-um:+.4f})")
print(f"personalization C  test MSE = {pc:.4f}   (delta {pc-um:+.4f})  <- submit")

## 5. Conclusions

1. Chronological splits + first-rating-day cold start prevent future leakage.
2. Under MSE, `user_mean` is the strongest baseline; ingredient-only Ridge adds little.
3. Personalization Model C (user_mean + HGB residual) reaches test MSE ≈ 0.3491, better than `user_mean` (0.3536) and Model A (0.3495).
4. To rebuild everything from scratch, set `RUN_FULL_PIPELINE = True` in Setup (requires raw CSVs for data prep, and several minutes for personalization).